# Step 12b — Balance the ground truth (cap N, then min per smell)

> Cap each `(smell, polarity)` bucket of the unbalanced ground truth to at most **N**, then trim each smell to `min(present, absent)` so every smell row is **symmetric** (Pos == Neg). This is the balanced human-validated ground truth.


## Inputs
| Path | Format | Source |
|---|---|---|
| `data/12_ground_truth_unbalanced.json` | JSON | Step 12 |
| `data/10_ground_truth_sample.json` | JSON | Step 10 (priority lookup) |
| `data/10b_ground_truth_resample.json` | JSON | Step 10b (priority lookup) |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/12b_final_ground_truth.json` | JSON (balanced ground truth, Step 12 schema) | downstream |
| `data/step12b_report.json` | JSON | audit |
| `data/12b_ground_truth_n{N}.json` | JSON (optional parametric-N variant) | analysis |



In [1]:
# --- Notebook bootstrap ---
from dataset_utils.notebook_bootstrap import setup

setup()

import sys
from pathlib import Path

# Guarded __file__ hack: make dataset_utils importable when run as a plain script
# without an installed wheel. Harmless in Jupyter (__file__ undefined). Do NOT remove.
try:
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
except NameError:
    pass

from dataset_utils.ground_truth_balancing import REPORT_PATH, N, run

print("Step 12b - balancing ground truth (cap N, then min per smell)...", flush=True)

# Balanced ground truth: cap each (smell, polarity) bucket to N, then trim each
# smell to min(present, absent) for an exact per-smell balance. The whole
# implementation lives in dataset_utils.ground_truth_balancing (so it is unit-
# tested and reused by the parametric cell below); this notebook only drives it.
output_path, report = run(n=N, balance_to_min=True)

print("Balanced final ground truth written.")
print(f"  Output : {output_path}")
print(f"  Report : {REPORT_PATH}")
print("-" * 48)
print(f"  Entries in     : {report['n_entries_in']}")
print(f"  Entries out    : {report['n_entries_out']}")
print(f"  Instances cut  : {report['instances_dropped']} (cap N={report['n_per_polarity_cap']})")
print(f"  balance_to_min : {report['balance_to_min']}")
print(f"  Smells < N     : {len(report['deficits_vs_cap'])}")


Step 12b - balancing ground truth (cap N, then min per smell)...


Balanced final ground truth written.
  Output : data/12b_final_ground_truth.json
  Report : data/step12b_report.json
------------------------------------------------
  Entries in     : 875
  Entries out    : 852
  Instances cut  : 149 (cap N=50)
  balance_to_min : True
  Smells < N     : 10


In [2]:
# --- Parametric-N alternative ground truths (optional) ---
# Emit data/12b_ground_truth_n{N}.json for each N in N_VARIANTS: the SAME
# cap-then-min path as the primary build above, just with a different
# per-polarity cap. Useful for sensitivity analysis (smaller, exactly-N
# balanced ground truths). These live in data/ root as first-class variants;
# old / non-min-balanced ground truths are archived under
# data/alternative_artifacts/.
from pathlib import Path

from dataset_utils.ground_truth_balancing import run

N_VARIANTS = [15]  # e.g. [15, 25, 40] to sweep; [] to skip

for n in N_VARIANTS:
    out_path = Path(f"data/12b_ground_truth_n{n}.json")
    rep_path = Path(f"data/step12b_n{n}_report.json")
    _, rep = run(output_path=out_path, report_path=rep_path, n=n, balance_to_min=True)
    print(
        f"N={n:<3} -> {out_path}  "
        f"(entries={rep['n_entries_out']}, cut={rep['instances_dropped']}, "
        f"smells<N={len(rep['deficits_vs_cap'])})"
    )


N=15  -> data/12b_ground_truth_n15.json  (entries=294, cut=1219, smells<N=1)


---

`12b_final_ground_truth.json` is the **base** balanced human-validated ground truth, consumed directly by the agent-evaluation module and the analysis layer for the 18 smells that need no synthesis.
